In [1]:
import json
import os
from IPython.display import display, HTML
from PIL import Image
import base64
from io import BytesIO
from collections import defaultdict

In [2]:
JSON_FILE_PATH = 'video_tsne_enriched_data_viz_without_base64.json'

# Load JSON data
with open(JSON_FILE_PATH, 'r') as f:
    data = json.load(f)

len(data)

1472

In [3]:
# Group entries by zero_shot_prediction
label_counts = defaultdict(int)
for item in data:
    label = item['payload']['zero_shot_prediction']
    label_counts[label] += 1

In [4]:
def image_to_base64(image_path):
    with Image.open(image_path) as img:
        buffered = BytesIO()
        img.save(buffered, format="JPEG")
        img_str = base64.b64encode(buffered.getvalue()).decode()
        return img_str

# Function to display thumbnails grid with hyperlinks for a given group
def display_thumbnails_group(title, items, columns=3):
    html = f'<h2 style="margin-top:30px;">{title}</h2>'
    html += '<table style="border-collapse: collapse;">'
    for i in range(0, len(items), columns):
        html += '<tr>'
        for item in items[i:i+columns]:
            video_id = item['payload']['video_id']
            thumbnail_path = item['payload']['thumbnail_path']
            video_path = item['payload']['video_path']

            # Construct localhost URL path
            filename = os.path.basename(video_path)
            video_url = f"http://localhost:8000/{filename}"

            # Convert image to base64
            try:
                img_base64 = image_to_base64(thumbnail_path)
            except FileNotFoundError:
                img_base64 = ""  # Skip if thumbnail not found

            # Add cell with image and hyperlink
            cell_html = f'''
                <td style="padding:10px; text-align:center;">
                    <a href="{video_url}" target="_blank">
                        <img src="data:image/jpeg;base64,{img_base64}" style="display:block; margin:auto; border:2px solid #000;" />
                    </a>
                    <div style="margin-top:5px;">ID {video_id}</div>
                    <div><a href="{video_url}" target="_blank">Link</a></div>
                </td>
            '''
            html += cell_html
        html += '</tr>'
    html += '</table>'

    display(HTML(html))

In [5]:
# for label, items in label_groups.items():
#     display_thumbnails_group(label, items, columns=3)

In [9]:
def display_thumbnails_for_label(label, columns=3):
    if label not in label_counts:
        print(f"No data found for label: {label}")
        return
    
    items = [item for item in data if item['payload']['zero_shot_prediction'] == label]
    # print(f"Displaying {len(items)} videos for label: {label}")
    
    # Display thumbnails grid
    display_thumbnails_group(label, items, columns=columns)

In [10]:
print("Available Labels and their Counts:")
for label, count in label_counts.items():
    print(f"- {label}: {count}")

Available Labels and their Counts:
- screen capture: 472
- dancing: 139
- DIY: 176
- animals: 106
- standup comedy: 107
- cooking: 90
- podcasts: 95
- shopping: 17
- fitness: 21
- monologue: 157
- adventure: 25
- talking head: 10
- miscellaneous: 25
- travel: 13
- water sports: 18
- home decor: 1


In [14]:
# display_thumbnails_for_label("animals")